In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Howard-style multiresolution ice benchmark
TWO DATASET VERSIONS FROM THE SAME ROBUSTLY CONVERGED TRAJECTORIES
================================================================

This generator produces BOTH benchmark variants in ONE simulation pass:

Version A:
    t=50 only
    100 LF training functions + 20 HF training functions
    50 held-out paired LF/HF test cases at t=50

Version B:
    trajectory snapshots t=1,...,50
    5000 LF training functions + 1000 HF training functions
    same 50 held-out paired LF/HF test trajectories, primary evaluation t=50

CRITICAL CONTROLLED-COMPARISON PROPERTY
---------------------------------------
Both masters are built from the EXACT SAME underlying simulated trajectories.
Therefore their t=50 states are bitwise identical.  The only difference is
whether the neural-network training dataset exposes only the final state or
all yearly snapshots.

ADAPTIVE PICARD CONVERGENCE
---------------------------
Every retained yearly velocity solve must satisfy picard_tol=1e-5.

Instead of changing the nonlinear equation, relaxation factor, or physical
parameters, this script uses adaptive Picard continuation:
    * run the usual max_picard_iters=30 block;
    * if not converged, continue from the current iterate with another block;
    * allow up to MAX_PICARD_BLOCKS blocks.

This preserves the same solver and fixed-point iteration while preventing a
trajectory from being contaminated by an unconverged intermediate solve.

The paper-style random field, grids, simulation budgets, and surrogate physics
are otherwise unchanged.

Outputs
-------
Folder:
    /kaggle/working/LAYER1_CANONICAL_gpScale0p35/

CANONICAL LAYER-1 master (use this for every Layer-1 experiment):
    howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz

A trajectory Version-B master is also written from the same simulations for archival purposes,
but it must NOT be used by the Layer-1 experiments.
"""

from __future__ import annotations

import json
import math
import time
import hashlib
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
from scipy.interpolate import RegularGridInterpolator
from scipy.sparse import coo_matrix, csc_matrix
from scipy.sparse.linalg import splu


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # Fixed random seeds: train and test are independent.
    train_seed: int = 20260827
    test_seed: int = 20260828

    # Howard multiresolution MF budgets.
    n_lf_train: int = 100
    n_hf_train: int = 20
    n_test: int = 50

    # Published random basal-friction setup.
    domain_m: float = 60_000.0
    beta_corr_m: float = 54_000.0
    beta_cov_amplitude: float = 1.0
    # Controlled difficulty knob: scale only the zero-mean GP perturbation
    # around log(beta_opt). s=1 reproduces the harder benchmark; here s=0.35.
    beta_gp_scale: float = 0.35
    beta_opt_base: float = 3000.0  # Pa yr / m

    # Published resolutions.
    n_lf: int = 15
    n_hf: int = 41

    # Trajectory benchmark: retain yearly snapshots t=1,...,50.
    final_year: int = 50
    evolution_step_years: float = 1.0

    # Reproduction-choice physical constants.
    # These are not all fully specified in the Howard paper.
    rho_ice: float = 910.0
    gravity: float = 9.8101
    glen_n: float = 3.0
    flow_A_per_year: float = 1.0e-16  # Pa^-3 yr^-1

    # Halfar-style initial dome used in our standalone benchmark.
    halfar_H0_m: float = 2000.0 * math.sqrt(0.125)
    halfar_R0_m: float = 60_000.0 * math.sqrt(0.125)

    # Reduced nonlinear membrane-style stress balance.
    # v2 convergence update: allow enough Picard iterations so the final
    # LF/HF states can actually satisfy picard_tol=1e-5.
    strain_reg_per_year: float = 1.0e-5
    viscosity_min_Pa_yr: float = 1.0e5
    viscosity_max_Pa_yr: float = 1.0e10
    max_picard_iters: int = 30
    picard_tol: float = 1.0e-5
    picard_relax: float = 0.65
    min_H_for_stress_m: float = 1.0

    # IMPORTANT: neutral/no-calibration values.
    membrane_scale: float = 1.0
    friction_response_power: float = 1.0

    # Thickness transport: dH/dt + div(H ubar) = 0.
    # We use zero external accumulation/forcing.
    cfl: float = 0.45
    max_transport_substep_yr: float = 0.20
    thickness_clip_max_m: float = 5000.0

    # Output.
    out_dir: str = "/kaggle/working/LAYER1_CANONICAL_gpScale0p35"
    master_name: str = "UNUSED_two_masters_written_explicitly.npz"
    dtype: str = "float32"


OVERWRITE_RUNS = False
OUT_DIR_OVERRIDE = None

# Continue the SAME Picard iteration in blocks of cfg.max_picard_iters.
# 5 x 30 = up to 150 iterations at a difficult yearly state.
MAX_PICARD_BLOCKS = 5
# Adaptive under-relaxation candidates for difficult transient states.
# The physical equations are unchanged; only nonlinear solver damping adapts.
PICARD_RELAX_CANDIDATES = (0.65, 0.50, 0.35, 0.25)


def build_config() -> Config:
    cfg = Config()
    if OUT_DIR_OVERRIDE is not None:
        cfg.out_dir = str(OUT_DIR_OVERRIDE)
    return cfg


# =============================================================================
# GRID / FIELD HELPERS
# =============================================================================

def grid_xy(n: int, L: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x = np.linspace(0.0, L, n, dtype=np.float64)
    y = np.linspace(0.0, L, n, dtype=np.float64)
    X, Y = np.meshgrid(x, y, indexing="xy")
    return x, y, X, Y


def beta_opt_field(X: np.ndarray, Y: np.ndarray, cfg: Config) -> np.ndarray:
    return cfg.beta_opt_base * (
        1.0
        + 0.5
        * np.sin(np.pi * X / 30_000.0)
        * np.sin(np.pi * Y / 30_000.0)
    )


def halfar_initial_thickness(X: np.ndarray, Y: np.ndarray, cfg: Config) -> np.ndarray:
    """
    Smooth radially symmetric Halfar-style dome used as a standalone
    reproduction choice.
    """
    xc = 0.5 * cfg.domain_m
    yc = 0.5 * cfg.domain_m
    r = np.sqrt((X - xc) ** 2 + (Y - yc) ** 2)

    n = cfg.glen_n
    q = np.maximum(
        0.0,
        1.0 - (r / cfg.halfar_R0_m) ** ((n + 1.0) / n),
    )
    H = cfg.halfar_H0_m * q ** (n / (2.0 * n + 1.0))
    return H.astype(np.float64)


def bilinear_resample(
    field: np.ndarray,
    x_src: np.ndarray,
    y_src: np.ndarray,
    x_dst: np.ndarray,
    y_dst: np.ndarray,
) -> np.ndarray:
    interp = RegularGridInterpolator(
        (y_src, x_src),
        field,
        method="linear",
        bounds_error=True,
    )
    Xd, Yd = np.meshgrid(x_dst, y_dst, indexing="xy")
    pts = np.column_stack([Yd.ravel(), Xd.ravel()])
    return interp(pts).reshape(len(y_dst), len(x_dst))


def grad2(f: np.ndarray, dx: float) -> Tuple[np.ndarray, np.ndarray]:
    fy, fx = np.gradient(f, dx, dx, edge_order=2)
    return fx, fy


def rel_l2(a: np.ndarray, b: np.ndarray, eps: float = 1.0e-12) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(
        np.linalg.norm(a.ravel() - b.ravel())
        / (np.linalg.norm(b.ravel()) + eps)
    )


# =============================================================================
# PAPER-STYLE GP beta SAMPLING
# =============================================================================

def build_or_load_beta_bank(cfg: Config, out_dir: Path) -> Dict[str, np.ndarray]:
    """
    Generate exactly 100 train beta realizations and 50 independent test
    realizations.

    All beta fields are sampled on the 41x41 grid from the paper-specified GP,
    then bilinearly evaluated on the 15x15 grid.  Therefore each LF/HF pair
    represents the same underlying random beta realization.
    """
    path = out_dir / "beta_train100_test50_gpScale0p35.npz"

    if path.exists():
        z = np.load(path, allow_pickle=False)
        valid = (
            int(z["n_train"]) == cfg.n_lf_train
            and int(z["n_test"]) == cfg.n_test
            and int(z["train_seed"]) == cfg.train_seed
            and int(z["test_seed"]) == cfg.test_seed
            and abs(float(z["beta_gp_scale"]) - cfg.beta_gp_scale) < 1.0e-12
        )
        if valid:
            print(f"[beta] Loading cached beta bank: {path}")
            return {k: z[k] for k in z.files}

    xh, yh, Xh, Yh = grid_xy(cfg.n_hf, cfg.domain_m)
    xl, yl, _, _ = grid_xy(cfg.n_lf, cfg.domain_m)

    pts = np.column_stack([Xh.ravel(), Yh.ravel()])
    sq = np.sum(pts * pts, axis=1)
    d2 = sq[:, None] + sq[None, :] - 2.0 * (pts @ pts.T)
    d2 = np.maximum(d2, 0.0)

    K = cfg.beta_cov_amplitude * np.exp(
        -0.5 * d2 / (cfg.beta_corr_m ** 2)
    )
    K.flat[:: K.shape[0] + 1] += 1.0e-10

    print(f"[beta] Controlled difficulty: log-beta perturbation scale s={cfg.beta_gp_scale}")
    print(
        f"[beta] Cholesky factorization of "
        f"{K.shape[0]} x {K.shape[0]} GP covariance ..."
    )
    Lc = np.linalg.cholesky(K)

    mean_gamma = np.log(beta_opt_field(Xh, Yh, cfg).ravel())

    def draw(seed: int, n_samples: int) -> np.ndarray:
        rng = np.random.default_rng(seed)
        Z = rng.standard_normal((K.shape[0], n_samples))
        gamma = mean_gamma[:, None] + cfg.beta_gp_scale * (Lc @ Z)
        return np.exp(gamma.T).reshape(
            n_samples, cfg.n_hf, cfg.n_hf
        )

    beta_hf_train100 = draw(cfg.train_seed, cfg.n_lf_train)
    beta_hf_test50 = draw(cfg.test_seed, cfg.n_test)

    beta_lf_train100 = np.empty(
        (cfg.n_lf_train, cfg.n_lf, cfg.n_lf),
        dtype=np.float64,
    )
    beta_lf_test50 = np.empty(
        (cfg.n_test, cfg.n_lf, cfg.n_lf),
        dtype=np.float64,
    )

    for i in range(cfg.n_lf_train):
        beta_lf_train100[i] = bilinear_resample(
            beta_hf_train100[i], xh, yh, xl, yl
        )

    for i in range(cfg.n_test):
        beta_lf_test50[i] = bilinear_resample(
            beta_hf_test50[i], xh, yh, xl, yl
        )

    payload = dict(
        beta_hf_train100=beta_hf_train100.astype(cfg.dtype),
        beta_lf_train100=beta_lf_train100.astype(cfg.dtype),
        beta_hf_test50=beta_hf_test50.astype(cfg.dtype),
        beta_lf_test50=beta_lf_test50.astype(cfg.dtype),
        x_hf=xh.astype(cfg.dtype),
        y_hf=yh.astype(cfg.dtype),
        x_lf=xl.astype(cfg.dtype),
        y_lf=yl.astype(cfg.dtype),
        train_seed=np.array(cfg.train_seed, dtype=np.int64),
        test_seed=np.array(cfg.test_seed, dtype=np.int64),
        n_train=np.array(cfg.n_lf_train, dtype=np.int64),
        n_test=np.array(cfg.n_test, dtype=np.int64),
        beta_gp_scale=np.array(cfg.beta_gp_scale, dtype=np.float64),
    )
    np.savez_compressed(path, **payload)
    print(f"[beta] Saved: {path}")
    return payload


# =============================================================================
# STANDALONE REDUCED MOLHO-LIKE / MEMBRANE-STYLE SOLVER
# =============================================================================

class ReducedIceFlowSolver:
    """
    Nonlinear 2-D membrane-style stress-balance approximation.

    Componentwise system:

        -div(K grad u) + beta u = -rho g H dH/dx
        -div(K grad v) + beta v = -rho g H dH/dy

    where

        K = 2 mu H
        mu = 1/2 A^(-1/n) De^(1/n - 1)

    The SAME solver is used at 15x15 and 41x41; only spatial resolution
    changes.  This mirrors the key multiresolution design in Howard Sec. 3.4.2.

    This is not a claim of exact equivalence to their FEniCS MOLHO model.
    """

    def __init__(self, n_nodes: int, cfg: Config):
        self.n = int(n_nodes)
        self.cfg = cfg
        self.L = float(cfg.domain_m)
        self.dx = self.L / (self.n - 1)
        self.ni = self.n - 2

        if self.ni <= 0:
            raise ValueError("Need at least 3 grid nodes.")

        self.Nint = self.ni * self.ni
        self.last_u = np.zeros(
            (2, self.n, self.n),
            dtype=np.float64,
        )

    def reset_velocity(self) -> None:
        self.last_u.fill(0.0)

    def _effective_viscosity(self, vel: np.ndarray) -> np.ndarray:
        u, v = vel[0], vel[1]
        ux, uy = grad2(u, self.dx)
        vx, vy = grad2(v, self.dx)

        De2 = (
            ux * ux
            + vy * vy
            + ux * vy
            + 0.25 * (uy + vx) ** 2
            + self.cfg.strain_reg_per_year ** 2
        )
        De = np.sqrt(
            np.maximum(
                De2,
                self.cfg.strain_reg_per_year ** 2,
            )
        )

        n = float(self.cfg.glen_n)
        A = float(self.cfg.flow_A_per_year)

        mu = (
            0.5
            * A ** (-1.0 / n)
            * De ** (1.0 / n - 1.0)
        )
        return np.clip(
            mu,
            self.cfg.viscosity_min_Pa_yr,
            self.cfg.viscosity_max_Pa_yr,
        )

    def _effective_beta(self, beta: np.ndarray) -> np.ndarray:
        # Paper-closer version: no artificial beta-response compression.
        p = float(self.cfg.friction_response_power)
        if abs(p - 1.0) > 1.0e-12:
            raise ValueError(
                "Paper-closer generator requires "
                "friction_response_power=1."
            )
        return np.asarray(beta, dtype=np.float64)

    def _idx(self, iy: int, ix: int) -> int:
        return (iy - 1) * self.ni + (ix - 1)

    def _build_matrix(
        self,
        K: np.ndarray,
        beta: np.ndarray,
    ) -> csc_matrix:
        rows = []
        cols = []
        vals = []
        h2 = self.dx * self.dx

        for iy in range(1, self.n - 1):
            for ix in range(1, self.n - 1):
                p = self._idx(iy, ix)

                ke = 0.5 * (K[iy, ix] + K[iy, ix + 1])
                kw = 0.5 * (K[iy, ix] + K[iy, ix - 1])
                kn = 0.5 * (K[iy, ix] + K[iy + 1, ix])
                ks = 0.5 * (K[iy, ix] + K[iy - 1, ix])

                diag = (
                    beta[iy, ix]
                    + (ke + kw + kn + ks) / h2
                )

                rows.append(p)
                cols.append(p)
                vals.append(diag)

                if ix + 1 <= self.n - 2:
                    rows.append(p)
                    cols.append(self._idx(iy, ix + 1))
                    vals.append(-ke / h2)

                if ix - 1 >= 1:
                    rows.append(p)
                    cols.append(self._idx(iy, ix - 1))
                    vals.append(-kw / h2)

                if iy + 1 <= self.n - 2:
                    rows.append(p)
                    cols.append(self._idx(iy + 1, ix))
                    vals.append(-kn / h2)

                if iy - 1 >= 1:
                    rows.append(p)
                    cols.append(self._idx(iy - 1, ix))
                    vals.append(-ks / h2)

        return coo_matrix(
            (vals, (rows, cols)),
            shape=(self.Nint, self.Nint),
        ).tocsc()

    def _rhs(
        self,
        H: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        Hx, Hy = grad2(H, self.dx)
        drive = self.cfg.rho_ice * self.cfg.gravity * H
        fx = -drive * Hx
        fy = -drive * Hy
        return (
            fx[1:-1, 1:-1].ravel(),
            fy[1:-1, 1:-1].ravel(),
        )

    def solve_velocity(
        self,
        H: np.ndarray,
        beta: np.ndarray,
    ) -> Tuple[np.ndarray, bool, int, float]:
        H = np.asarray(H, dtype=np.float64)
        beta = np.asarray(beta, dtype=np.float64)

        if H.shape != (self.n, self.n):
            raise ValueError("H grid shape mismatch.")
        if beta.shape != (self.n, self.n):
            raise ValueError("beta grid shape mismatch.")

        beta_eff = self._effective_beta(beta)
        vel = self.last_u.copy()

        # Local basal-drag initial guess.
        if not np.any(vel):
            Hx, Hy = grad2(H, self.dx)
            denom = np.maximum(beta_eff, 1.0)
            vel[0] = (
                -self.cfg.rho_ice
                * self.cfg.gravity
                * H
                * Hx
                / denom
            )
            vel[1] = (
                -self.cfg.rho_ice
                * self.cfg.gravity
                * H
                * Hy
                / denom
            )
            vel[:, 0, :] = 0.0
            vel[:, -1, :] = 0.0
            vel[:, :, 0] = 0.0
            vel[:, :, -1] = 0.0

        rhs_u, rhs_v = self._rhs(H)
        final_rel = np.inf
        converged = False

        for nit in range(1, self.cfg.max_picard_iters + 1):
            mu = self._effective_viscosity(vel)
            Hstress = np.maximum(
                H,
                self.cfg.min_H_for_stress_m,
            )

            K = (
                self.cfg.membrane_scale
                * 2.0
                * mu
                * Hstress
            )

            M = self._build_matrix(K, beta_eff)
            lu = splu(M)

            uint = lu.solve(rhs_u)
            vint = lu.solve(rhs_v)

            new = np.zeros_like(vel)
            new[0, 1:-1, 1:-1] = uint.reshape(
                self.ni, self.ni
            )
            new[1, 1:-1, 1:-1] = vint.reshape(
                self.ni, self.ni
            )

            mask = H <= 1.0e-6
            new[:, mask] = 0.0

            final_rel = (
                np.linalg.norm((new - vel).ravel())
                / (np.linalg.norm(new.ravel()) + 1.0e-12)
            )

            if final_rel < self.cfg.picard_tol:
                vel = new
                converged = True
                break

            # IMPORTANT:
            # Even on the final allowed iteration, preserve the under-relaxed
            # iterate.  This makes repeated Picard blocks a true continuation
            # rather than jumping to the unrelaxed fixed-point map at every
            # block boundary.
            a = self.cfg.picard_relax
            vel = a * new + (1.0 - a) * vel

        # Reaching max iterations is not treated as a fatal error; we record it.
        self.last_u = vel.copy()
        return vel, converged, nit, float(final_rel)


# =============================================================================
# THICKNESS TRANSPORT
# =============================================================================

def upwind_transport_step(
    H: np.ndarray,
    u: np.ndarray,
    v: np.ndarray,
    dt: float,
    dx_m: float,
) -> np.ndarray:
    """
    Conservative update for

        dH/dt + div(H ubar) = 0

    corresponding to zero external accumulation/forcing.
    """
    ny, nx = H.shape

    ux = 0.5 * (u[:, :-1] + u[:, 1:])
    Hx_up = np.where(
        ux >= 0.0,
        H[:, :-1],
        H[:, 1:],
    )
    Fx_int = ux * Hx_up
    Fx = np.zeros((ny, nx + 1), dtype=np.float64)
    Fx[:, 1:nx] = Fx_int

    vy = 0.5 * (v[:-1, :] + v[1:, :])
    Hy_up = np.where(
        vy >= 0.0,
        H[:-1, :],
        H[1:, :],
    )
    Fy_int = vy * Hy_up
    Fy = np.zeros((ny + 1, nx), dtype=np.float64)
    Fy[1:ny, :] = Fy_int

    div = (
        (Fx[:, 1:] - Fx[:, :-1]) / dx_m
        + (Fy[1:, :] - Fy[:-1, :]) / dx_m
    )

    Hn = H - dt * div

    # Fixed square domain; zero thickness on outer boundary.
    Hn[0, :] = 0.0
    Hn[-1, :] = 0.0
    Hn[:, 0] = 0.0
    Hn[:, -1] = 0.0
    return Hn


def advance_thickness(
    H: np.ndarray,
    vel: np.ndarray,
    years: float,
    dx_m: float,
    cfg: Config,
) -> np.ndarray:
    u, v = vel[0], vel[1]

    vmax = max(
        float(np.max(np.abs(u))),
        float(np.max(np.abs(v))),
        1.0e-12,
    )

    dt_cfl = cfg.cfl * dx_m / vmax
    dt = min(
        float(years),
        cfg.max_transport_substep_yr,
        dt_cfl,
    )

    nsub = max(
        1,
        int(math.ceil(years / max(dt, 1.0e-12))),
    )
    dt = years / nsub

    Hn = H.copy()
    for _ in range(nsub):
        Hn = upwind_transport_step(
            Hn,
            u,
            v,
            dt,
            dx_m,
        )
        Hn = np.clip(
            Hn,
            0.0,
            cfg.thickness_clip_max_m,
        )
    return Hn


# =============================================================================
# TRAJECTORY DATASET
# =============================================================================

# =============================================================================
# TRAJECTORY SIMULATION / CACHE
# =============================================================================

def run_cache_path(
    cache_dir: Path,
    fidelity: str,
    split: str,
    idx: int,
) -> Path:
    return cache_dir / f"{fidelity}_{split}_{idx:04d}.npz"


def solve_velocity_robust(
    solver: ReducedIceFlowSolver,
    H: np.ndarray,
    beta_grid: np.ndarray,
    cfg: Config,
    context: str,
) -> Tuple[np.ndarray, bool, int, float, int, float]:
    """
    Robust Picard solve with adaptive under-relaxation.

    Why this is needed
    ------------------
    Some transient thickness states are more nonlinear than the final t=50
    state.  At those states alpha=0.65 can fail to be a contraction and the
    Picard residual may stagnate even after many iterations.

    Strategy
    --------
    1) Start from the previous converged velocity field.
    2) Try the normal alpha=0.65.
    3) If it fails, RESET to the same original initial guess and retry with
       stronger damping: 0.50, 0.35, then 0.25.
    4) Each candidate may continue for several 30-iteration blocks.

    This changes only nonlinear-solver damping; beta, H, the PDE surrogate,
    physical constants, grids, and convergence tolerance are unchanged.

    Returns
    -------
    vel, converged, total_iters_for_successful_candidate, rel,
    blocks_used_for_successful_candidate, relaxation_used
    """
    start_vel = solver.last_u.copy()
    original_relax = float(cfg.picard_relax)

    try:
        for alpha in PICARD_RELAX_CANDIDATES:
            # Every candidate gets the SAME initial guess, so a failed aggressive
            # iteration does not contaminate the more strongly damped retry.
            solver.last_u = start_vel.copy()
            cfg.picard_relax = float(alpha)

            total_iters = 0
            rel = np.inf
            vel = None

            for block in range(1, MAX_PICARD_BLOCKS + 1):
                vel, conv, nit, rel = solver.solve_velocity(H, beta_grid)
                total_iters += int(nit)

                if conv:
                    # solver.last_u already contains the converged velocity.
                    return (
                        vel,
                        True,
                        total_iters,
                        float(rel),
                        block,
                        float(alpha),
                    )

        # Restore the original starting state before raising.
        solver.last_u = start_vel.copy()
        raise RuntimeError(
            "Adaptive Picard failed for "
            f"{context}. Tried relaxations={PICARD_RELAX_CANDIDATES}, "
            f"{MAX_PICARD_BLOCKS} blocks x {cfg.max_picard_iters} iterations "
            "per relaxation."
        )
    finally:
        # Restore the configured default for the next state. The converged
        # velocity remains stored in solver.last_u.
        cfg.picard_relax = original_relax


def simulate_trajectory(
    solver: ReducedIceFlowSolver,
    beta_grid: np.ndarray,
    cfg: Config,
    fidelity: str,
    split: str,
    idx: int,
    cache_path: Path,
) -> Dict[str, np.ndarray]:
    """
    Evolve one beta realization and retain yearly snapshots t=1,...,50.

    Each retained pair is:
        input  = (beta, H(t))
        target = U(t)

    Every retained velocity solve must satisfy picard_tol. If an intermediate
    snapshot does not converge within max_picard_iters, the run stops instead
    of silently placing an unconverged field into the ML dataset.
    """
    if cache_path.exists() and not OVERWRITE_RUNS:
        z = np.load(cache_path, allow_pickle=False)
        if "H_traj" in z.files and "U_traj" in z.files:
            print(f"[resume] {cache_path.name}")
            return {k: z[k] for k in z.files}

    n = solver.n
    _, _, X, Y = grid_xy(n, cfg.domain_m)
    H = halfar_initial_thickness(X, Y, cfg)
    solver.reset_velocity()

    # Initial t=0 velocity is only needed to advance H to t=1.
    vel, conv, nit, rel, blocks, relax_used = solve_velocity_robust(
        solver,
        H,
        beta_grid,
        cfg,
        context=f"{fidelity} {split} {idx} at t=0",
    )

    n_steps = int(round(cfg.final_year / cfg.evolution_step_years))
    if abs(n_steps * cfg.evolution_step_years - cfg.final_year) > 1.0e-12:
        raise ValueError("final_year must be divisible by evolution_step_years.")

    H_snap = np.empty((n_steps, n, n), dtype=np.float32)
    U_snap = np.empty((n_steps, 2, n, n), dtype=np.float32)
    conv_snap = np.empty((n_steps,), dtype=np.int8)
    nit_snap = np.empty((n_steps,), dtype=np.int32)
    rel_snap = np.empty((n_steps,), dtype=np.float64)
    blocks_snap = np.empty((n_steps,), dtype=np.int16)
    relax_snap = np.empty((n_steps,), dtype=np.float32)
    times = np.arange(1, n_steps + 1, dtype=np.float32) * np.float32(
        cfg.evolution_step_years
    )

    for k in range(n_steps):
        H = advance_thickness(
            H,
            vel,
            cfg.evolution_step_years,
            solver.dx,
            cfg,
        )

        vel, conv, nit, rel, blocks, relax_used = solve_velocity_robust(
            solver,
            H,
            beta_grid,
            cfg,
            context=f"{fidelity} {split} {idx} at t={times[k]:.1f}",
        )

        if not np.isfinite(H).all():
            raise FloatingPointError(
                f"Non-finite H in {fidelity} {split} {idx} at t={times[k]:.1f}."
            )
        if not np.isfinite(vel).all():
            raise FloatingPointError(
                f"Non-finite U in {fidelity} {split} {idx} at t={times[k]:.1f}."
            )
        H_snap[k] = H.astype(np.float32)
        U_snap[k] = vel.astype(np.float32)
        conv_snap[k] = int(conv)
        nit_snap[k] = int(nit)
        rel_snap[k] = float(rel)
        blocks_snap[k] = int(blocks)
        relax_snap[k] = np.float32(relax_used)

    payload = dict(
        beta=beta_grid.astype(np.float32),
        times=times,
        H_traj=H_snap,
        U_traj=U_snap,
        converged_traj=conv_snap,
        picard_iters_traj=nit_snap,
        picard_rel_traj=rel_snap,
        picard_blocks_traj=blocks_snap,
        picard_relax_traj=relax_snap,
        H_t50=H_snap[-1],
        U_t50=U_snap[-1],
        sample_idx=np.array(idx, dtype=np.int64),
        fidelity=np.array(fidelity),
        split=np.array(split),
    )

    np.savez_compressed(cache_path, **payload)

    speed50 = np.sqrt(
        (payload["U_t50"].astype(np.float64) ** 2).sum(axis=0)
    )
    print(
        f"[done] {cache_path.name} | "
        f"H50max={float(payload['H_t50'].max()):.2f} m | "
        f"U50max={float(speed50.max()):.3f} m/yr | "
        f"worstSnapshotRel={float(rel_snap.max()):.2e} | "
        f"maxBlocks={int(blocks_snap.max())} | "
        f"minRelax={float(relax_snap.min()):.2f}"
    )
    return payload


# =============================================================================
# GENERATE LF100 / HF20 / PAIRED TEST50 TRAJECTORIES
# =============================================================================

def generate_all_runs(
    cfg: Config,
    bank: Dict[str, np.ndarray],
    out_dir: Path,
) -> None:
    cache = out_dir / "runs"
    cache.mkdir(parents=True, exist_ok=True)

    beta_lf_train = bank["beta_lf_train100"].astype(np.float64)
    beta_hf_train100 = bank["beta_hf_train100"].astype(np.float64)
    beta_lf_test = bank["beta_lf_test50"].astype(np.float64)
    beta_hf_test = bank["beta_hf_test50"].astype(np.float64)

    print("\n=== LF TRAIN100 TRAJECTORIES: 15x15, retain t=1..50 ===")
    lf_solver = ReducedIceFlowSolver(cfg.n_lf, cfg)
    for i in range(cfg.n_lf_train):
        simulate_trajectory(
            lf_solver, beta_lf_train[i], cfg, "LF", "train", i,
            run_cache_path(cache, "LF", "train", i),
        )

    print("\n=== HF TRAIN20 TRAJECTORIES: paired beta cases 0..19, retain t=1..50 ===")
    hf_solver = ReducedIceFlowSolver(cfg.n_hf, cfg)
    for i in range(cfg.n_hf_train):
        simulate_trajectory(
            hf_solver, beta_hf_train100[i], cfg, "HF", "train", i,
            run_cache_path(cache, "HF", "train", i),
        )

    print("\n=== LF TEST50 TRAJECTORIES: retain t=1..50 ===")
    lf_solver = ReducedIceFlowSolver(cfg.n_lf, cfg)
    for i in range(cfg.n_test):
        simulate_trajectory(
            lf_solver, beta_lf_test[i], cfg, "LF", "test", i,
            run_cache_path(cache, "LF", "test", i),
        )

    print("\n=== HF TEST50 TRAJECTORIES: retain t=1..50 ===")
    hf_solver = ReducedIceFlowSolver(cfg.n_hf, cfg)
    for i in range(cfg.n_test):
        simulate_trajectory(
            hf_solver, beta_hf_test[i], cfg, "HF", "test", i,
            run_cache_path(cache, "HF", "test", i),
        )


def load_run(
    cache: Path,
    fidelity: str,
    split: str,
    idx: int,
) -> Dict[str, np.ndarray]:
    path = run_cache_path(cache, fidelity, split, idx)
    if not path.exists():
        raise FileNotFoundError(path)
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}


def stack_trajectory_runs(
    cache: Path,
    fidelity: str,
    split: str,
    n: int,
) -> Tuple[
    np.ndarray, np.ndarray, np.ndarray,
    np.ndarray, np.ndarray, np.ndarray
]:
    beta, H, U, conv, rel, relax = [], [], [], [], [], []
    for i in range(n):
        d = load_run(cache, fidelity, split, i)
        beta.append(d["beta"])
        H.append(d["H_traj"])
        U.append(d["U_traj"])
        conv.append(d["converged_traj"])
        rel.append(d["picard_rel_traj"])
        relax.append(d["picard_relax_traj"])
    return (
        np.stack(beta),
        np.stack(H),
        np.stack(U),
        np.stack(conv),
        np.stack(rel),
        np.stack(relax),
    )


# =============================================================================
# MASTER TRAJECTORY DATASET
# =============================================================================

def assemble_both_masters(
    cfg: Config,
    out_dir: Path,
) -> Tuple[Path, Path]:
    cache = out_dir / "runs"

    beta_lf_train, H_lf_train_traj, U_lf_train_traj, conv_lf_train, rel_lf_train, relax_lf_train = (
        stack_trajectory_runs(cache, "LF", "train", cfg.n_lf_train)
    )
    beta_hf_train, H_hf_train_traj, U_hf_train_traj, conv_hf_train, rel_hf_train, relax_hf_train = (
        stack_trajectory_runs(cache, "HF", "train", cfg.n_hf_train)
    )
    beta_lf_test, H_lf_test_traj, U_lf_test_traj, conv_lf_test, rel_lf_test, relax_lf_test = (
        stack_trajectory_runs(cache, "LF", "test", cfg.n_test)
    )
    beta_hf_test, H_hf_test_traj, U_hf_test_traj, conv_hf_test, rel_hf_test, relax_hf_test = (
        stack_trajectory_runs(cache, "HF", "test", cfg.n_test)
    )

    x_lf, y_lf, _, _ = grid_xy(cfg.n_lf, cfg.domain_m)
    x_hf, y_hf, _, _ = grid_xy(cfg.n_hf, cfg.domain_m)
    times = np.arange(1, cfg.final_year + 1, dtype=np.float32)

    common_metadata = asdict(cfg)
    common_metadata.update(
        {
            "paper_reference": (
                "Howard et al., Journal of Computational Physics "
                "493 (2023) 112462, Sec. 3.4.2"
            ),
            "simulation_budget": (
                "100 LF train trajectories + 20 HF train trajectories + "
                "50 paired LF/HF test trajectories"
            ),
            "sampling_protocol": (
                "random GP realizations; fixed seeds; no k-center; "
                f"log-beta perturbation scale s={cfg.beta_gp_scale}"
            ),
            "solver_protocol": (
                "same reduced membrane-style solver; adaptive Picard under-relaxation "
                "alpha in {0.65,0.50,0.35,0.25}; every retained yearly "
                "solve required rel<1e-5"
            ),
            "critical_control": (
                "Version A and Version B are sliced from the exact same "
                "underlying trajectory arrays; t=50 fields are identical"
            ),
        }
    )

    # -----------------------
    # Version A: t=50 only.
    # -----------------------
    meta_a = dict(common_metadata)
    meta_a.update(
        {
            "version": "A_t50_only",
            "training_protocol": "one t=50 function per training trajectory",
            "N_L_training_functions": 100,
            "N_H_training_functions": 20,
            "primary_test": "50 held-out HF trajectories at t=50",
        }
    )

    path_a = out_dir / "howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz"
    np.savez_compressed(
        path_a,
        x_lf=x_lf.astype(np.float32),
        y_lf=y_lf.astype(np.float32),
        x_hf=x_hf.astype(np.float32),
        y_hf=y_hf.astype(np.float32),
        time=np.array(cfg.final_year, dtype=np.float32),

        beta_lf_train=beta_lf_train,
        H_lf_train=H_lf_train_traj[:, -1],
        U_lf_train=U_lf_train_traj[:, -1],

        beta_hf_train=beta_hf_train,
        H_hf_train=H_hf_train_traj[:, -1],
        U_hf_train=U_hf_train_traj[:, -1],

        beta_lf_test=beta_lf_test,
        H_lf_test=H_lf_test_traj[:, -1],
        U_lf_test=U_lf_test_traj[:, -1],

        beta_hf_test=beta_hf_test,
        H_hf_test=H_hf_test_traj[:, -1],
        U_hf_test=U_hf_test_traj[:, -1],

        metadata_json=np.array(json.dumps(meta_a, indent=2)),
    )

    # -----------------------
    # Version B: t=1..50.
    # -----------------------
    meta_b = dict(common_metadata)
    meta_b.update(
        {
            "version": "B_trajectory_t1_50",
            "training_protocol": "all yearly snapshots t=1,...,50",
            "N_L_training_functions": 5000,
            "N_H_training_functions": 1000,
            "primary_test": "50 held-out HF trajectories at t=50",
        }
    )

    path_b = out_dir / "howard_ice_MF_VERSION_B_trajectory_t1_50_gpScale0p35_adaptivePicard.npz"
    np.savez_compressed(
        path_b,
        x_lf=x_lf.astype(np.float32),
        y_lf=y_lf.astype(np.float32),
        x_hf=x_hf.astype(np.float32),
        y_hf=y_hf.astype(np.float32),
        times=times,

        beta_lf_train=beta_lf_train,
        H_lf_train_traj=H_lf_train_traj,
        U_lf_train_traj=U_lf_train_traj,

        beta_hf_train=beta_hf_train,
        H_hf_train_traj=H_hf_train_traj,
        U_hf_train_traj=U_hf_train_traj,

        beta_lf_test=beta_lf_test,
        H_lf_test_traj=H_lf_test_traj,
        U_lf_test_traj=U_lf_test_traj,

        beta_hf_test=beta_hf_test,
        H_hf_test_traj=H_hf_test_traj,
        U_hf_test_traj=U_hf_test_traj,

        H_lf_train_t50=H_lf_train_traj[:, -1],
        U_lf_train_t50=U_lf_train_traj[:, -1],
        H_hf_train_t50=H_hf_train_traj[:, -1],
        U_hf_train_t50=U_hf_train_traj[:, -1],
        H_lf_test_t50=H_lf_test_traj[:, -1],
        U_lf_test_t50=U_lf_test_traj[:, -1],
        H_hf_test_t50=H_hf_test_traj[:, -1],
        U_hf_test_t50=U_hf_test_traj[:, -1],

        converged_lf_train=conv_lf_train,
        converged_hf_train=conv_hf_train,
        converged_lf_test=conv_lf_test,
        converged_hf_test=conv_hf_test,
        picard_rel_lf_train=rel_lf_train,
        picard_rel_hf_train=rel_hf_train,
        picard_rel_lf_test=rel_lf_test,
        picard_rel_hf_test=rel_hf_test,

        picard_relax_lf_train=relax_lf_train,
        picard_relax_hf_train=relax_hf_train,
        picard_relax_lf_test=relax_lf_test,
        picard_relax_hf_test=relax_hf_test,

        metadata_json=np.array(json.dumps(meta_b, indent=2)),
    )

    # Bitwise control check: A t=50 must be exactly B last snapshot.
    za = np.load(path_a, allow_pickle=False)
    zb = np.load(path_b, allow_pickle=False)
    exact_checks = [
        np.array_equal(za["H_lf_train"], zb["H_lf_train_t50"]),
        np.array_equal(za["U_lf_train"], zb["U_lf_train_t50"]),
        np.array_equal(za["H_hf_train"], zb["H_hf_train_t50"]),
        np.array_equal(za["U_hf_train"], zb["U_hf_train_t50"]),
        np.array_equal(za["H_hf_test"], zb["H_hf_test_t50"]),
        np.array_equal(za["U_hf_test"], zb["U_hf_test_t50"]),
    ]
    if not all(exact_checks):
        raise RuntimeError("Version A/B t=50 identity check FAILED.")

    with open(out_dir / "metadata_version_A.json", "w") as f:
        json.dump(meta_a, f, indent=2)
    with open(out_dir / "metadata_version_B.json", "w") as f:
        json.dump(meta_b, f, indent=2)

    return path_a, path_b


# =============================================================================
# SANITY REPORT
# =============================================================================

def sanity_report(
    path_a: Path,
    path_b: Path,
    cfg: Config,
) -> None:
    a = np.load(path_a, allow_pickle=False)
    b = np.load(path_b, allow_pickle=False)

    print("\n" + "=" * 112)
    print("TWO-VERSION ROBUST DATASET SANITY REPORT")
    print("=" * 112)

    print("Version A master :", path_a.name)
    print("Version B master :", path_b.name)
    print("A LF/HF training :", len(a["H_lf_train"]), len(a["H_hf_train"]))
    print(
        "B LF/HF snapshots:",
        b["H_lf_train_traj"].shape[0] * b["H_lf_train_traj"].shape[1],
        b["H_hf_train_traj"].shape[0] * b["H_hf_train_traj"].shape[1],
    )

    exact = [
        np.array_equal(a["H_lf_train"], b["H_lf_train_t50"]),
        np.array_equal(a["U_lf_train"], b["U_lf_train_t50"]),
        np.array_equal(a["H_hf_train"], b["H_hf_train_t50"]),
        np.array_equal(a["U_hf_train"], b["U_hf_train_t50"]),
        np.array_equal(a["H_hf_test"], b["H_hf_test_t50"]),
        np.array_equal(a["U_hf_test"], b["U_hf_test_t50"]),
    ]
    print("A/B t=50 fields bitwise identical :", "PASS" if all(exact) else "FAIL")

    conv_names = [
        ("LF train", b["converged_lf_train"], b["picard_rel_lf_train"]),
        ("HF train", b["converged_hf_train"], b["picard_rel_hf_train"]),
        ("LF test", b["converged_lf_test"], b["picard_rel_lf_test"]),
        ("HF test", b["converged_hf_test"], b["picard_rel_hf_test"]),
    ]
    all_conv = True
    for name, c, r in conv_names:
        ok = int(c.sum())
        total = int(c.size)
        all_conv &= (ok == total)
        print(
            f"{name:10s} all yearly solves converged : "
            f"{ok}/{total} | worst rel={float(r.max()):.3e}"
        )
    print("All yearly Picard solves converged :", "PASS" if all_conv else "FAIL")

    relax_sets = [
        ("LF train", b["picard_relax_lf_train"]),
        ("HF train", b["picard_relax_hf_train"]),
        ("LF test", b["picard_relax_lf_test"]),
        ("HF test", b["picard_relax_hf_test"]),
    ]
    for name, arr in relax_sets:
        fallback = int(np.sum(arr < 0.65 - 1.0e-7))
        print(
            f"{name:10s} snapshots using stronger damping : "
            f"{fallback}/{arr.size} | min alpha={float(arr.min()):.2f}"
        )

    print("Version A training protocol         : t=50 only")
    print("Version B training protocol         : t=1..50")
    print("Underlying trajectories             : EXACTLY THE SAME")
    print("=" * 112)


# =============================================================================
# MAIN
# =============================================================================

def print_spec(cfg: Config) -> None:
    print("=" * 108)
    print("HOWARD-STYLE CONTROLLED-DIFFICULTY DATASET | GP SCALE s=0.35")
    print("=" * 108)
    print("Same seeds / physics / grids / budgets as the harder s=1 Version A.")
    print(f"Only difficulty change: log-beta GP perturbation scale s={cfg.beta_gp_scale}.")
    print("Difference: retain yearly snapshots for ML training.")
    print("Simulation budgets:")
    print("  LF train trajectories = 100")
    print("  HF train trajectories = 20 (paired with LF trajectories 0..19)")
    print("  paired test trajectories = 50")
    print("Training-function counts if every snapshot is used:")
    print("  LF = 100 x 50 = 5000")
    print("  HF = 20  x 50 = 1000")
    print("Primary evaluation: held-out HF at t=50.")
    print("No k-center, no shared-H, no response compression, no calibration scale.")
    print(f"Output folder : {cfg.out_dir}")
    print("Masters       : Version A t=50 + Version B t=1..50")
    print("=" * 108)


def main() -> None:
    cfg = build_config()

    if cfg.n_hf_train > cfg.n_lf_train:
        raise ValueError("Need n_hf_train <= n_lf_train.")
    if abs(cfg.friction_response_power - 1.0) > 1.0e-12:
        raise ValueError("Trajectory paper-closer dataset requires friction_response_power=1.")
    if abs(cfg.membrane_scale - 1.0) > 1.0e-12:
        raise ValueError("Trajectory paper-closer dataset requires membrane_scale=1.")
    if abs(cfg.evolution_step_years - 1.0) > 1.0e-12:
        raise ValueError("This trajectory version expects yearly snapshots (dt_output=1 yr).")
    if abs(cfg.beta_gp_scale - 0.35) > 1.0e-12:
        raise ValueError("This script is locked to beta_gp_scale=0.35 for the controlled benchmark.")

    out_dir = Path(cfg.out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    print_spec(cfg)
    t0 = time.time()

    bank = build_or_load_beta_bank(cfg, out_dir)
    generate_all_runs(cfg, bank, out_dir)
    path_a, path_b = assemble_both_masters(cfg, out_dir)
    sanity_report(path_a, path_b, cfg)

    sha256 = hashlib.sha256(Path(path_a).read_bytes()).hexdigest()
    print("\n" + "=" * 108)
    print("LAYER-1 CANONICAL DATASET READY")
    print("USE THIS FILE FOR EVERY LAYER-1 EXPERIMENT:")
    print(path_a)
    print("SHA256:", sha256)
    print("beta_gp_scale: 0.35")
    print("Version B is archival only; do NOT use it for Layer 1.")
    print("=" * 108)
    print(f"Saved Version B archival master: {path_b}")
    print(f"Elapsed wall time: {(time.time() - t0)/60.0:.2f} min")


if __name__ == "__main__":
    main()


HOWARD-STYLE CONTROLLED-DIFFICULTY DATASET | GP SCALE s=0.35
Same seeds / physics / grids / budgets as the harder s=1 Version A.
Only difficulty change: log-beta GP perturbation scale s=0.35.
Difference: retain yearly snapshots for ML training.
Simulation budgets:
  LF train trajectories = 100
  HF train trajectories = 20 (paired with LF trajectories 0..19)
  paired test trajectories = 50
Training-function counts if every snapshot is used:
  LF = 100 x 50 = 5000
  HF = 20  x 50 = 1000
Primary evaluation: held-out HF at t=50.
No k-center, no shared-H, no response compression, no calibration scale.
Output folder : /kaggle/working/LAYER1_CANONICAL_gpScale0p35
Masters       : Version A t=50 + Version B t=1..50
[beta] Controlled difficulty: log-beta perturbation scale s=0.35
[beta] Cholesky factorization of 1681 x 1681 GP covariance ...
[beta] Saved: /kaggle/working/LAYER1_CANONICAL_gpScale0p35/beta_train100_test50_gpScale0p35.npz

=== LF TRAIN100 TRAJECTORIES: 15x15, retain t=1..50 ===
[do